In [1]:
print("hi")

hi


In [2]:
from graph import GraphState
from src.agents.intent_agent import Filter
from src.agents.intent_agent import IntentState, Join, Aggregation


# def planner_node(state: GraphState):
def planner_node(state: IntentState):
    """
    Resolves raw IntentState into SQL-executable IntentState.
    """

    intent = state.intent

    if intent is None:
        raise ValueError("Planner received empty intent.")

    resolved_filters = []

    # for f in intent.filters:
    for f in state.filters:

        # RULE 1: aggregation-based filter → HAVING
        if f.aggregation:
            resolved_filters.append(f)
            continue

        # RULE 2: derived filter (subquery exists, no aggregation)
        if f.subquery and f.value is None:
            # Explicitly mark as derived filter
            resolved_filters.append(
                Filter(
                    column=f.column,
                    operator=f.operator,
                    value=f.subquery,          # ignored downstream
                    aggregation=None,
                    subquery=f.subquery  # must be used by generator
                )
            )
            continue

        # RULE 3: normal row-level filter
        if f.value is not None:
            resolved_filters.append(f)
            continue

        # RULE 4: unresolved filter → reject
        raise ValueError(
            f"Unresolvable filter detected: {f}. "
            "Planner cannot produce executable intent."
        )

    # Replace filters with resolved ones
    # resolved_intent = intent.model_copy(
    #     update={"filters": resolved_filters}
    # )
    state.filters = resolved_filters

    return state
    # return state.model_copy(update={"intent": resolved_intent})


intent='SELECT' tables=['users', 'purchases'] columns=['users.id', 'users.name'] joins=[Join(table1='users', table2='purchases', column1='id', column2='user_id')] filters=[Filter(column='purchases.amount', operator='>', value=None, aggregation='AVG', subquery=IntentState(intent='SELECT', tables=['purchases'], columns=[], joins=[], filters=[], group_by=[], order_by=[], aggregations=[Aggregation(column='amount', function='AVG')], limit=None))] group_by=['users.id', 'users.name'] order_by=[] aggregations=[] limit=None
intent='SELECT' tables=['users', 'purchases'] columns=['users.id', 'users.name'] joins=[Join(table1='users', table2='purchases', column1='id', column2='user_id')] filters=[Filter(column='purchases.amount', operator='>', value=None, aggregation='AVG', subquery=IntentState(intent='SELECT', tables=['purchases'], columns=[], joins=[], filters=[], group_by=[], order_by=[], aggregations=[Aggregation(column='amount', function='AVG')], limit=None))] group_by=['users.id', 'users.name

In [ ]:
intent_state = IntentState(intent='SELECT',
                            tables=['users', 'purchases'],
                            columns=['users.id', 'users.name'],
                            joins=[Join(table1='users', table2='purchases', column1='id', column2='user_id')],
                            filters=[Filter(column='purchases.amount', operator='>', value=None, aggregation=None, subquery=IntentState(intent='SELECT', tables=['purchases'], columns=[], joins=[], filters=[], group_by=[], order_by=[], aggregations=[Aggregation(column='amount', function='AVG')], limit=None))],
                            group_by=[],
                            order_by=[],
                            aggregations=[],
                            limit=None)

res = planner_node(intent_state)
print(res)

intent='SELECT' tables=['users', 'purchases'] columns=['users.id', 'users.name'] joins=[Join(table1='users', table2='purchases', column1='id', column2='user_id')] filters=[Filter(column='purchases.amount', operator='>', value=IntentState(intent='SELECT', tables=['purchases'], columns=[], joins=[], filters=[], group_by=[], order_by=[], aggregations=[Aggregation(column='amount', function='AVG')], limit=None), aggregation=None, subquery=IntentState(intent='SELECT', tables=['purchases'], columns=[], joins=[], filters=[], group_by=[], order_by=[], aggregations=[Aggregation(column='amount', function='AVG')], limit=None))] group_by=[] order_by=[] aggregations=[] limit=None


In [4]:
from src.agents.sql_generator import SQLGeneratorAgent

gen = SQLGeneratorAgent()
res2 = gen.generate(res)
print(res2)

SELECT users.id, users.name
FROM users
JOIN purchases ON users.id = purchases.user_id
WHERE purchases.amount > intent='SELECT' tables=['purchases'] columns=[] joins=[] filters=[] group_by=[] order_by=[] aggregations=[Aggregation(column='amount', function='AVG')] limit=None;
